In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score
from scipy.stats import norm

In [0]:
# Display all columns
pd.set_option('display.max_columns', None)

In [0]:
# create sample data that has three columns y-pred, y-actual and gender
import pandas as pd
import numpy as np

# Generate sample data
n = 100000  # number of rows
np.random.seed(42)
y_true=np.random.randint(0, 2, size=n)
age=np.random.randint(0, 100, n)

predicted = []
for a, ag in zip(y_true, age):
    if ag < 17:  # Introduce bias: lower accuracy for younger group
        if np.random.rand() > 0.4:  # Only 60% correct for age < 17
            predicted.append(a)
        else:
            predicted.append(1 - a)
    elif ag > 75:  # Introduce bias: lower accuracy for older group
        if np.random.rand() > 0.26:  # 70% correct for age >75
            predicted.append(a)
        else:
            predicted.append(1 - a)
    else:  # model is more accurate
        if np.random.rand() > 0.2:  #80% correct for age < 17
            predicted.append(a)
        else:
            predicted.append(1 - a)

df = pd.DataFrame({
    'y_pred': predicted,
    'y_true': y_true,
    'age': age
})
print(df.head())

   y_pred  y_true  age
0       1       0    8
1       1       1   99
2       0       0   69
3       0       0   23
4       0       0   89


In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score
from scipy.stats import norm

def bootstrap_ci(data, metric_fn, n_bootstrap=5000, alpha=0.05):
    """Compute metric and its CI using bootstrapping."""
    stats = []
    for _ in range(n_bootstrap):
        sample = data.sample(frac=1, replace=True)
        stats.append(metric_fn(sample))
    ci_low = np.percentile(stats, 100 * (alpha/2))
    ci_high = np.percentile(stats, 100 * (1 - alpha/2))
    return np.mean(stats), (ci_low, ci_high), stats

def cohen_d(a, b):
    """Compute Cohen's d for effect size."""
    m1, m2 = np.mean(a), np.mean(b)
    s1, s2 = np.std(a, ddof=1), np.std(b, ddof=1)
    s = np.sqrt(((len(a)-1)*s1**2 + (len(b)-1)*s2**2) / (len(a)+len(b)-2))
    return (m1 - m2) / s if s > 0 else 0.0

# Example: Assume df has columns 'y_true', 'y_pred', 'age_group'
def precision_metric(group_df):
    return precision_score(group_df['y_true'], group_df['y_pred'])

def fairness_analysis(df, sensitive_col='age_group'):
    results = []
    # Global metric & distribution
    global_mean, global_ci, global_stats = bootstrap_ci(df, precision_metric)
    # For each subgroup
    for group, group_df in df.groupby(sensitive_col):
        sub_mean, sub_ci, sub_stats = bootstrap_ci(group_df, precision_metric)
        bias_flag = False
        effect_size = None
        # Step 1: CI Check
        if not (sub_ci[0] <= global_mean <= sub_ci[1]):
            # Step 2: Effect Size
            effect_size = abs(cohen_d(np.array(global_stats), np.array(sub_stats)))
            if effect_size >= 0.5:  # Medium or Large effect
                bias_flag = True
        results.append({
            sensitive_col: group,
            'subgroup_precision': sub_mean,
            'subgroup_ci': sub_ci,
            'bias_flag': bias_flag,
            'effect_size': effect_size,
            'global_precision': global_mean
        })
    return pd.DataFrame(results)

age_labels = ['0-17', '18-29', '30-44', '45-59', '60-74', '75+']
age_bins = [0, 18, 30, 45, 60, 75, 100]
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)
result_df = fairness_analysis(df, sensitive_col='age_group')
# print CI interval and bias flag
print(result_df)

  age_group  subgroup_precision                               subgroup_ci  \
0      0-17            0.619560   (0.609499304143327, 0.6297439175881858)   
1     18-29            0.793273  (0.7830029774057818, 0.8033483217433067)   
2     30-44            0.795707  (0.7865796428803646, 0.8049562832713405)   
3     45-59            0.803710  (0.7947080218698817, 0.8126701062283916)   
4     60-74            0.799863   (0.7908071611838163, 0.809139345349717)   
5       75+            0.749607  (0.7421203265590886, 0.7570560970188468)   

   bias_flag  effect_size  global_precision  
0       True    34.543342          0.753703  
1       True    10.099293          0.753703  
2       True    11.787623          0.753703  
3       True    14.108700          0.753703  
4       True    12.922615          0.753703  
5      False          NaN          0.753703  
